### Gas Cylinder Fine-Tuning

Fine-tunes YOLO11n on the available gas-cylinder datasets and keeps one run per dataset.

Datasets:
- `datasets/gas_bottles/` — https://universe.roboflow.com/klearchos-stavrothanasopoulos-konstantinos-gkountakos-6jwgj/cylinders-iaq6n/dataset/3
- `datasets/gas_cylinder/` — https://universe.roboflow.com/obj-dect/gas-cylinder-detection/dataset/3

Both datasets are trained with the same settings so results can be compared. Runs go into separate folders so they never overwrite each other.

Outputs per dataset:
- training run directory under `results/gas_bottle_training/yolo11n_<dataset>/`
- `weights/best.pt` and `weights/last.pt`
- `results.csv` written by Ultralytics
- training plots written by Ultralytics

In [11]:
from pathlib import Path
import pandas as pd
import torch
import yaml
from ultralytics import YOLO

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

device = 0 if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

PyTorch: 2.14.1+cpu
CUDA available: False
Using device: cpu


### Inputs and outputs

**Input datasets:** `datasets/gas_bottles/` and `datasets/gas_cylinder/`.

**Output runs:** `results/gas_bottle_training/yolo11n_<dataset>/`.

Each run contains:
- `weights/best.pt`, `weights/last.pt`
- `results.csv` (Ultralytics writes this automatically)
- `results.png`, `labels.jpg`, and other plots written by Ultralytics

In [12]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATASETS_DIR = ROOT / "datasets"
RUNS_DIR = ROOT / "results" / "gas_bottle_training"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Datasets folder: {DATASETS_DIR.relative_to(ROOT)}  exists={DATASETS_DIR.exists()}")
print(f"Runs folder:     {RUNS_DIR.relative_to(ROOT)}")

Datasets folder: datasets  exists=True
Runs folder:     results\gas_bottle_training


In [15]:
print("=" * 70)
print("Dataset Checking")
print("=" * 70)

dataset_dirs = sorted(
    p for p in DATASETS_DIR.iterdir()
    if p.is_dir() and any((p / s).exists() for s in ["train", "valid", "test"])
)

if not dataset_dirs:
    print(f"No dataset folders found under {DATASETS_DIR.relative_to(ROOT)}")
else:
    for ds in dataset_dirs:
        print(f"\n{ds.name}")
        print("-" * 70)

        for split in ["train", "valid", "test"]:
            lbl_dir = ds / split / "labels"
            img_dir = ds / split / "images"

            if not lbl_dir.exists():
                continue

            labels = list(lbl_dir.glob("*.txt"))
            images = list(img_dir.glob("*")) if img_dir.exists() else []

            counts = {}
            empty = 0
            for lbl in labels:
                text = lbl.read_text().strip()
                if not text:
                    empty += 1
                    continue
                for line in text.splitlines():
                    parts = line.split()
                    if parts:
                        cls = int(parts[0])
                        counts[cls] = counts.get(cls, 0) + 1

            total_boxes = sum(counts.values())
            per_class = "  ".join(f"c{k}={v}" for k, v in sorted(counts.items()))

            print(f"  {split:6s} images={len(images):5d}  label_files={len(labels):5d}  "
                  f"boxes={total_boxes:5d}  empty_labels={empty:3d}  {per_class}")

        # Quick flags
        train_lbl = ds / "train" / "labels"
        val_lbl = ds / "valid" / "labels"
        n_train = len(list(train_lbl.glob("*.txt"))) if train_lbl.exists() else 0
        n_val = len(list(val_lbl.glob("*.txt"))) if val_lbl.exists() else 0

        if n_train == 0:
            print("  ⚠ no training labels")
        if n_val == 0:
            print("  ⚠ no validation labels")
        if 0 < n_train < 200:
            print(f"  ⚠ only {n_train} training images — metrics will be noisy")
        if 0 < n_val < 50:
            print(f"  ⚠ only {n_val} validation images — metrics will be unstable")

print("\n" + "=" * 70)

Dataset Checking

gas_bottles
----------------------------------------------------------------------
  train  images=18137  label_files=18137  boxes=18137  empty_labels=  0  c0=18137
  valid  images= 4862  label_files= 4862  boxes= 4862  empty_labels=  0  c0=4862
  test   images= 2270  label_files= 2270  boxes= 2270  empty_labels=  0  c0=2270

gas_cylinder
----------------------------------------------------------------------
  train  images=   80  label_files=   80  boxes=  210  empty_labels=  0  c0=109  c1=101
  valid  images=    8  label_files=    8  boxes=   24  empty_labels=  0  c0=13  c1=11
  test   images=   20  label_files=   20  boxes=   64  empty_labels=  0  c0=37  c1=27
  ⚠ only 80 training images — metrics will be noisy
  ⚠ only 8 validation images — metrics will be unstable



### Dataset inventory

Checks each dataset, its YAML, and the number of images per split.

In [3]:
DATASET_CANDIDATES = [
    "gas_bottles",
    "gas_cylinder",
]


def load_yaml(ds_dir):
    for name in ["data.yaml", "data_ultralytics.yaml"]:
        p = ds_dir / name
        if p.exists():
            return p, yaml.safe_load(p.read_text())
    return None, None


print("=" * 60)
print("DATASET INVENTORY")
print("=" * 60)

for name in DATASET_CANDIDATES:
    ds = DATASETS_DIR / name
    if not ds.exists():
        print(f"{name:15s} MISSING")
        continue

    yaml_path, cfg = load_yaml(ds)
    if yaml_path is None:
        print(f"{name:15s} no yaml found")
        continue

    n_train = len(list((ds / "train" / "images").glob("*"))) if (ds / "train" / "images").exists() else 0
    n_valid = len(list((ds / "valid" / "images").glob("*"))) if (ds / "valid" / "images").exists() else 0
    n_test = len(list((ds / "test" / "images").glob("*"))) if (ds / "test" / "images").exists() else 0

    print(f"{name:15s} yaml={yaml_path.name}  nc={cfg.get('nc')}  names={cfg.get('names')}")
    print(f"{'':15s} train={n_train} valid={n_valid} test={n_test}")

print("=" * 60)

DATASET INVENTORY
gas_bottles     yaml=data.yaml  nc=1  names=['gas_cylinder']
                train=18137 valid=4862 test=2270
gas_cylinder    yaml=data.yaml  nc=2  names=['Recognising-Indian-Indian-gas', 'gas cylinder head']
                train=80 valid=8 test=20


### Training configuration

Same settings for every dataset so runs are comparable. Small image size, small batch, few epochs. Laptop-friendly.

Defaults are conservative:
- **epochs:** 5
- **image size:** 416
- **batch size:** 4
- **fraction:** 0.25 (only 25% of training images per epoch — change to 1.0 for a full run)

`fraction` exists because `gas_bottles` has ~18,000 training images. Without it, one epoch on CPU would take hours.

In [4]:
EPOCHS = 5
IMGSZ = 416
BATCH = 4
FRACTION = 0.25
WORKERS = 0
SEED = 0

print(f"epochs={EPOCHS}  imgsz={IMGSZ}  batch={BATCH}  fraction={FRACTION}  workers={WORKERS}  seed={SEED}")
print(f"device={device}")

epochs=5  imgsz=416  batch=4  fraction=0.25  workers=0  seed=0
device=cpu


### Train every dataset

Loops over the datasets. For each one:
1. Loads its YAML.
2. Trains YOLO11n with the settings above.
3. Saves the run to `results/gas_bottle_training/yolo11n_<dataset>/`.

Training output and per-epoch validation metrics are printed by Ultralytics as it runs.

In [5]:
trained = []

for name in DATASET_CANDIDATES:
    ds = DATASETS_DIR / name
    if not ds.exists():
        print(f"[skip] {name} missing")
        continue

    yaml_path, cfg = load_yaml(ds)
    if yaml_path is None:
        print(f"[skip] {name} has no yaml")
        continue

    run_name = f"yolo11n_{name}"
    print("\n" + "=" * 60)
    print(f"Training {name}  ->  run name: {run_name}")
    print(f"nc={cfg.get('nc')}  names={cfg.get('names')}")
    print("=" * 60)

    model = YOLO("yolo11n.pt")

    model.train(
        data=str(yaml_path),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        fraction=FRACTION,
        workers=WORKERS,
        device=device,
        project=str(RUNS_DIR),
        name=run_name,
        exist_ok=True,
        plots=True,
        verbose=True,
        val=True,
        seed=SEED,
    )

    trained.append(name)

print(f"\nFinished training on: {trained}")


Training gas_bottles  ->  run name: yolo11n_gas_bottles
nc=1  names=['gas_cylinder']
New https://pypi.org/project/ultralytics/8.4.181 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.175  Python-3.11.9 torch-2.14.1+cpu CPU (AMD Ryzen 7 6800HS with Radeon Graphics)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=d:\BCS Files\Y3S1\AI Applications MBI38j\Labs\primagaz-yolo\datasets\gas_bottles\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=0.25, freeze=None, hsv_h=0.015, hsv_

### Results

In [6]:
def pick(row, *keys, default=None):
    for k in keys:
        for c in row.index:
            if c.lower() == k.lower():
                return row[c]
    return default


summaries = []

for name in trained:
    run_dir = RUNS_DIR / f"yolo11n_{name}"
    results_csv = run_dir / "results.csv"

    if not results_csv.exists():
        print(f"[skip] {name}: no results.csv")
        continue

    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]
    last = df.iloc[-1]

    _, cfg = load_yaml(DATASETS_DIR / name)

    summaries.append({
        "dataset": name,
        "classes": str(cfg.get("names")) if cfg else None,
        "nc": cfg.get("nc") if cfg else None,
        "precision": pick(last, "metrics/precision(B)", "precision"),
        "recall": pick(last, "metrics/recall(B)", "recall"),
        "mAP50": pick(last, "metrics/mAP50(B)", "mAP50"),
        "mAP50-95": pick(last, "metrics/mAP50-95(B)", "mAP50-95"),
    })

df_summary = pd.DataFrame(summaries)

### Overview and saved weights

Summary table plus a quick check that `best.pt` and `last.pt` exist for each run.

In [16]:
def pick(row, *keys, default=None):
    for k in keys:
        for c in row.index:
            if c.lower() == k.lower():
                return row[c]
    return default


def find_dataset_dir(name):
    for candidate in [name, name.replace("_v2", ""), name.replace("_v3", "")]:
        ds = DATASETS_DIR / candidate
        if ds.exists():
            return ds
    return None


summaries = []

run_folders = sorted(
    p for p in RUNS_DIR.iterdir()
    if p.is_dir() and (p / "results.csv").exists()
)

for run_dir in run_folders:
    dataset_name = run_dir.name.replace("yolo11n_", "")

    df = pd.read_csv(run_dir / "results.csv")
    df.columns = [c.strip() for c in df.columns]
    last = df.iloc[-1]

    ds = find_dataset_dir(dataset_name)
    _, cfg = load_yaml(ds) if ds else (None, None)

    best_pt = run_dir / "weights" / "best.pt"
    last_pt = run_dir / "weights" / "last.pt"

    summaries.append({
        "dataset": dataset_name,
        "classes": str(cfg.get("names")) if cfg else None,
        "nc": cfg.get("nc") if cfg else None,
        "precision": pick(last, "metrics/precision(B)", "precision"),
        "recall": pick(last, "metrics/recall(B)", "recall"),
        "mAP50": pick(last, "metrics/mAP50(B)", "mAP50"),
        "mAP50-95": pick(last, "metrics/mAP50-95(B)", "mAP50-95"),
        "best_pt": best_pt.exists(),
        "last_pt": last_pt.exists(),
        "run_dir": str(run_dir.relative_to(ROOT)),
    })

df_summary = pd.DataFrame(summaries)

print("=" * 90)
print("TRAINING SUMMARY (last-epoch metrics, all runs)")
print("=" * 90)
if df_summary.empty:
    print("No runs found under", RUNS_DIR.relative_to(ROOT))
else:
    print(df_summary.drop(columns=["run_dir"]).to_string(index=False))
print("=" * 90)

print("\nRun folders:")
for row in summaries:
    print(f"  {row['dataset']:20s} best={row['best_pt']}  last={row['last_pt']}  -> {row['run_dir']}")

print("\n" + "=" * 90)
print("INPUTS AND OUTPUTS")
print("=" * 90)
print(f"Input datasets:   {DATASETS_DIR.relative_to(ROOT)}/")
print(f"Output runs:      {RUNS_DIR.relative_to(ROOT)}/yolo11n_<dataset>/")
print(f"Runs found:       {len(run_folders)}")
print(f"Datasets trained: {[r['dataset'] for r in summaries]}")

TRAINING SUMMARY (last-epoch metrics, all runs)
     dataset                                                classes  nc  precision  recall   mAP50  mAP50-95  best_pt  last_pt
 gas_bottles                                       ['gas_cylinder']   1    0.99142 0.81743 0.94292   0.91486     True     True
gas_cylinder ['Recognising-Indian-Indian-gas', 'gas cylinder head']   2    0.68123 0.26923 0.29569   0.17583     True     True

Run folders:
  gas_bottles          best=True  last=True  -> results\gas_bottle_training\yolo11n_gas_bottles
  gas_cylinder         best=True  last=True  -> results\gas_bottle_training\yolo11n_gas_cylinder

INPUTS AND OUTPUTS
Input datasets:   datasets/
Output runs:      results\gas_bottle_training/yolo11n_<dataset>/
Runs found:       2
Datasets trained: ['gas_bottles', 'gas_cylinder']
